## ACNS 2026 Tutorial: Bumps uncertainty modelling for physical scientists

**inverse problems**

In this notebook we will look at the the inverse problem of inferring uncertainty in parameters $ω$ when we have a set of measurements $y_k$ at $x_k$ and a model $f(x, ω) = y$.

**Bayesian theorem**

Inverse problems can be analyzed using Bayes Theorem:

> $P(ω | y) = P(y | w) P(w) / \int_Ω P(y | w) P(w) \,\mathrm{d}ω$

The parts of the equation have names:

* prior $P(ω)$<br>
The prior distribution includes all of the information known about a parameter. If you are using a value from the literature in your model, you can incorporate the uncertainty in the value in your prior.

* likelihood $P(y | ω)$<br>
This is the joint probability of observing all $y_k$ values for a parameter set $ω$, giving $P(y | x, ω) = \prod_k P(y_k | x_k, ω)$


* posterior $P(ω | y)$<br>
The posterior distribution contains all the prior information about the parameters updated with new information from the measurement. From this we can extract statistics on the individual parameters and their correlations.

* model evidence $\int_Ω P(y | ω) P(ω) \,\mathrm{d}ω = P(y)$<br>
This is the likelihood of seeing the measured data integrated over the entire parameter space. In that sense, it is the probability that the model can be used to describe the data, hence the term "model evidence".

**χ² fitting**

If the measurement uncertainty is gaussian, with $y = f(x) + ε$ for $ε \sim \text{N}(0, σ^2)$, then the likelihood is just 

> $P(y | x, ω) = e^{-χ^2/2}/C$

with normalizing constant $C = \prod_k \sqrt{2πσ_k^2}$.

Traditional χ² minimization is equivalent to maximum likelihood estimation on the posterior, but only for normally distributed measurement uncertainties and uniform priors on the parameters. The uncertainty estimate using the covariance matrix assumes the posterior is gaussian.

**Bayesian analysis**

Bayesian methods allows:
* prior information about the parameters
* complex constraints between the parameters
* non-gaussian measurement uncertainty
* simultaneous fitting of related measurements
* non-gaussian posterior distributions


In [ ]:
# Uncomment the following line to install packages used within this notebook
%pip install numpy scipy uncertainties bumps matplotlib plotly

**bumps**

The program *Bumps* represents all problems using the "negative loglikelihood" or *nllf*.

> *nllf* = log likelihood + log prior

Normalizing constants may be missing because the existing algorithms don't care.

Various traditional optimizers such as Nelder-Mead simplex, Differential Evolution, Levenberg-Marquardt, BFGS

DREAM is a Markov Chain Monte Carlo (MCMC) sampler 

In [ ]:
# Prepare the environment
import bumps.names as bp
import numpy as np

%matplotlib inline
bp.help()

In [ ]:
# Define a simple fit problem y=mx + b

# Exercises:
# - change formula (e.g, x/m + b)
# - change priors (e.g., M.m.dev())

# Bumps parameter priors
# par.range(low, high) sets x ~ U[low, high]
# par.pmp(δ) sets x ~ U[p ± δ/100]
# par.pm(δ) sets x ~ [p-δ, p+δ]
# par.pm(δm, δp) sets x ~ U[p-δm, p+δp]
# par.dev(σ) sets the x ~ N(p, σ²)
#    if mean is given, use x ~ N(μ, σ²)
#    if limits=[low, high] are given, truncate the gaussian
# par.pdf(dist) sets x ~ dist where dist is a scipy.stats distribution

# data
x = [1, 2, 3, 4, 5, 6]
y = [2.1, 4.0, 6.3, 8.03, 9.6, 11.9]
dy = [0.05, 0.05, 0.2, 0.05, 0.2, 0.2]

# function
def line(x, m=1, b=0):
    return m * x + b

# model = function + data
M = bp.Curve(line, x, y, dy, m=2, b=2)
M.m.range(0, 4)
M.b.range(-5, 5)

# problem is a set of models
problem = bp.FitProblem(M)

In [ ]:
# Run a dream fit
options = dict(fit="dream", burn=0, samples=30000, verbose=True)
result = bp.fit(problem, **options)
bp.show_results(problem, result)

In [ ]:
# Poisson example

# Exercise: explore other gaussian approximations
#   x ~ N(x+1, √(x+1))         Expected value
#   x ~ N(x+0.5, √(x + 0.25))  Pearson
#   x ~ N(x, √(x+1))           Expected MLE
#   x ~ N(x, √x); x=0 ~ N(0.5, 0.5)  Pearson zero
#   x ~ N(x, √x); x=0 ~ N(0, 1) Expected zero

def peak(x, scale, center, width, background):
    return scale * np.exp(-0.5 * (x - center) ** 2 / width**2) + background

# data
scale, center, width, background = 3, 12, 1.5, 1
npoints = 145
x = np.linspace(5, 20, npoints)
y = np.random.poisson(peak(x, scale, center, width, background))
dy = np.sqrt(y)
dy[y==0] = 1.0

dx = max(x) - min(x)
MP = bp.PoissonCurve(peak, x, y, scale=1, center=2, width=2, background=0)
MP.scale.range(0, max(y) * 1.5)
MP.center.range(min(x) - 0.2 * dx, max(x) + 0.2 * dx)
MP.width.range(0, 0.7 * dx)
MP.background.range(0, max(y))

poisson_peak = bp.FitProblem(MP, name="Poisson Peak")

MG = bp.Curve(peak, x, y, dy, scale=1, center=2, width=2, background=0)
MG.scale.range(0, max(y) * 1.5)
MG.center.range(min(x) - 0.2 * dx, max(x) + 0.2 * dx)
MG.width.range(0, 0.7 * dx)
MG.background.range(0, max(y))

gauss_peak = bp.FitProblem(MG, name="Gaussian Peak")
#gauss_peak.show()

In [ ]:
# Run a dream fit for poisson
options = dict(fit="dream", burn=500, samples=30000, verbose=False)
poisson_result = bp.fit(poisson_peak, **options)
#bp.show_results(poisson_peak, poisson_result)

In [ ]:
# Run a dream fit for gauss
options = dict(fit="dream", burn=500, samples=30000, verbose=False)
gauss_result = bp.fit(gauss_peak, **options)
#bp.show_results(gauss_peak, gauss_result)

In [ ]:
print(f"target model: {scale=} {center=} {width=} {background=}")
bp.show_table(poisson_peak, poisson_result)
bp.show_table(gauss_peak, gauss_result)


In [ ]:
bp.show_results(poisson_peak, poisson_result)

In [ ]:
# Do we trust bumps as a global optimizer?

"""
introduce the multimodal gaussian mixture model

show that dream can sample from it

explain the various plots

talk about what good trace, convergence, logp, corner plats and parameter histgorams look like
"""

from bumps.util import push_seed

def make_model(dims=10):
    """
    Adjust S (sigma^2) to change the diameter of the peak

    Small values of sigma lead to stuck fits.

    Adjust relative intensity of peaks with I.

    Choose peak spacing with x. Keep it between -10 and 20

    The peaks are located in a hypercube of dimension d, with coordinates in
    each dimension a random permutation of x. That means the resulting histogram
    which is a projection along that dimension will contain a random permutation
    of peak heights, with centers at the x positions. The 2D histograms will be
    be similar random permutations.
    """
    from bumps.dream.model import Mixture, MVNormal

    if 1:  # Uneven spacing of 5 peaks
        num_modes = 5
        S = [0.1] * 5
        x = [1, 3, 7, 14, 18]
        I = [2.5, 1, 5, 4, 1]
        # S[2] = 0.05; I[2] = 60
    elif 1:  # Even spacing of 5 peaks
        num_modes = 5
        S = [0.1] * 5
        x = [-4, -2, 0, 2, 4]
        I = [15, 2.5, 1, 4, 1]
    else:  # Lots of evenly spaced peaks
        num_modes = 40
        S = [0.1] * num_modes
        x = np.linspace(-9, 9, num_modes)
        I = 2 * np.linspace(-1, 1, num_modes) ** 2 + 1

    print("generated peaks")
    print("   position =", x)
    print("   scale =", I)
    print("   width =", S)

    centers = [x] + [np.random.permutation(x) for _ in range(1, dims)]
    centers = np.asarray(centers).T
    args = []  # Sequence of density, weight, density, weight, ...
    for mu_i, Si, Ii in zip(centers, S, I):
        args.extend((MVNormal(mu_i, Si * np.eye(dims)), Ii))
    model = Mixture(*args)

    if 1:
        from bumps.dream.entropy import GaussianMixture

        pairs = zip(args[0::2], args[1::2])
        triples = ((M.mu, M.sigma, I) for M, I in pairs)
        mu, sigma, weight = zip(*triples)
        D = GaussianMixture(weight, mu=mu, sigma=sigma)
        print("*** Expected entropy: %s bits" % (D.entropy(N=100000) / np.log(2),))

    return model

def plot2d(fn, args=None, range=(-10, 10)):
    """
    Return a mesh plotter for the given function.

    *args* are the function arguments that are to be meshed (usually the
    first two arguments to the function).  *range* is the bounding box
    for the 2D mesh.

    All arguments except the meshed arguments are held fixed.
    """
    if args is None:
        args = [0, 1]

    def plotter(p, view=None):
        import matplotlib.pyplot as plt
        if len(p) == 1:
            x = p[0]
            r = np.linspace(range[0], range[1], 400)
            plt.plot(x + r, [fn(v) for v in x + r])
            plt.xlabel(args[0])
            plt.ylabel("-log P(%s)" % args[0])
        else:
            r = np.linspace(range[0], range[1], 20)
            x, y = p[args[0]], p[args[1]]
            data = np.empty((len(r), len(r)), "d")
            for j, xj in enumerate(x + r):
                for k, yk in enumerate(y + r):
                    p[args[0]], p[args[1]] = xj, yk
                    data[j, k] = fn(p)
            plt.pcolormesh(x + r, y + r, data)
            plt.plot(
                x, y, "o", markersize=6, markerfacecolor="red", markeredgecolor="black", markeredgewidth=1, alpha=0.7
            )
            plt.xlabel(args[0])
            plt.ylabel(args[1])

    return plotter

# Need reproducible models if we want to be able to resume a fit
# Note: modern way is to pass a random number generator
dims = 6
with push_seed(1):
    model = make_model(dims=dims)

M = bp.VectorPDF(model.nllf, p=[1.0] * dims, plot=plot2d(model.nllf))
for _, p in M.parameters().items():
    p.range(-10, 20)
multimodal = bp.FitProblem(M, name="Gaussian Mixture")

In [ ]:
# Run a dream fit
options = dict(fit="dream", burn=1000, samples=30000, verbose=False)
result = bp.fit(multimodal, **options)
bp.show_results(multimodal, result)

other topics:

* bic and model selection
* nested sampling and bayes factors
* experiment design with fisher information (entropy of covariance matrix)
* cluster computing: watch the fit while it runs remotely using ssh tunnels
* parallel tempering
* systematic error and nuisance parameters
* constraints, simultaneous fitting, priors

